In [ ]:
# import libraries

import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
import category_encoders as ce 
from sklearn import metrics
import matplotlib.pyplot as plt 
import seaborn as sns 
import warnings
import hyperopt
from scipy.stats import skew, norm
from hyperopt import hp, tpe, fmin, Trials
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.neighbors import KNeighborsRegressor
import os

import warnings
warnings.filterwarnings(action="ignore")

plt.style.use('default')

In [ ]:
# load data
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

train_data = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_data = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
train_data.columns = train_data.columns.str.replace(' ', '')
test_data.columns = test_data.columns.str.replace(' ', '')

target = train_data['SalePrice']
id = test_data['Id']
train_data = train_data.drop(['SalePrice', 'Id'], axis=1)
test_data = test_data.drop(['Id'], axis=1)

train_test = pd.concat([train_data, test_data], axis=0, sort=False)

In [ ]:
# detect NaN values

nan = pd.DataFrame(train_test.isna().sum(), columns = ['NaN_sum'])
nan['feat'] = nan.index
nan['Perc(%)'] = (nan['NaN_sum']/1460)*100
nan = nan[nan['NaN_sum'] > 0]
nan = nan.sort_values(by = ['NaN_sum'])
nan['Usability'] = np.where(nan['Perc(%)'] > 20, 'Discard', 'Keep')
nan

In [ ]:
# converting non-numeric predictors stored as numbers into string

train_test['MSSubClass'] = train_test['MSSubClass'].apply(str)
train_test['YrSold'] = train_test['YrSold'].apply(str)
train_test['MoSold'] = train_test['MoSold'].apply(str)

# filling categorical NaN 

train_test['Functional'] = train_test['Functional'].fillna('Typ')
train_test['Electrical'] = train_test['Electrical'].fillna("SBrkr")
train_test['KitchenQual'] = train_test['KitchenQual'].fillna("TA")
train_test['Exterior1st'] = train_test['Exterior1st'].fillna(train_test['Exterior1st'].mode()[0])
train_test['Exterior2nd'] = train_test['Exterior2nd'].fillna(train_test['Exterior2nd'].mode()[0])
train_test['SaleType'] = train_test['SaleType'].fillna(train_test['SaleType'].mode()[0])
train_test["PoolQC"] = train_test["PoolQC"].fillna("None")
train_test["Alley"] = train_test["Alley"].fillna("None")
train_test['FireplaceQu'] = train_test['FireplaceQu'].fillna("None")
train_test['Fence'] = train_test['Fence'].fillna("None")
train_test['MiscFeature'] = train_test['MiscFeature'].fillna("None")

for col in ('GarageArea', 'GarageCars'):
    train_test[col] = train_test[col].fillna(0)
        
for col in ['GarageType', 'GarageFinish', 'GarageQual', 'GarageCond']:
    train_test[col] = train_test[col].fillna('None')
    
for col in ('BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2'):
    train_test[col] = train_test[col].fillna('None')
    
    # checking the features with NaN remained out

for col in train_test:
    if train_test[col].isna().sum() > 0:
        print(train_test[col][0])

In [ ]:
# removing the useless variables

useless = ['GarageYrBlt','YearRemodAdd'] 
train_test = train_test.drop(useless, axis = 1)

# imputing with KnnRegressor

def impute_knn(df):
    ttn = train_test.select_dtypes(include=[np.number])
    ttc = train_test.select_dtypes(exclude=[np.number])

    cols_nan = ttn.columns[ttn.isna().any()].tolist()
    cols_no_nan = ttn.columns.difference(cols_nan).values

    for col in cols_nan:
        imp_test = ttn[ttn[col].isna()] 
        imp_train = ttn.dropna()
        model = KNeighborsRegressor(n_neighbors=5)
        knr = model.fit(imp_train[cols_no_nan], imp_train[col])
        ttn.loc[ttn[col].isna(), col] = knr.predict(imp_test[cols_no_nan])
    
    return pd.concat([ttn,ttc],axis=1)

train_test = impute_knn(train_test)


objects = []
for i in train_test.columns:
    if train_test[i].dtype == object:
        objects.append(i)
train_test.update(train_test[objects].fillna('None'))

# # Checking NaN presence

for col in train_test:
    if train_test[col].isna().sum() > 0:
        print(train_test[col][0])

In [ ]:
# feature engineering

train_test["SqFtPerRoom"] = train_test["GrLivArea"] / (train_test["TotRmsAbvGrd"] +
                                                       train_test["FullBath"] +
                                                       train_test["HalfBath"] +
                                                       train_test["KitchenAbvGr"])

train_test['Total_Home_Quality'] = train_test['OverallQual'] + train_test['OverallCond']

train_test['Total_Bathrooms'] = (train_test['FullBath'] + (0.5 * train_test['HalfBath']) +
                               train_test['BsmtFullBath'] + (0.5 * train_test['BsmtHalfBath']))

train_test["HighQualSF"] = train_test["1stFlrSF"] + train_test["2ndFlrSF"]

# converting non-numeric predictors stored as numbers into string

train_test['MSSubClass'] = train_test['MSSubClass'].apply(str)
train_test['YrSold'] = train_test['YrSold'].apply(str)
train_test['MoSold'] = train_test['MoSold'].apply(str)

# creating dummy variables from categorical features

train_test_dummy = pd.get_dummies(train_test)

# fetch all numeric features

numeric_features = train_test_dummy.select_dtypes(include=[np.number]).columns
numeric_features = train_test_dummy.dtypes[train_test_dummy.dtypes.isin([np.int64, np.float64])].index
skewed_features = train_test_dummy[numeric_features].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
high_skew = skewed_features[skewed_features > 0.5]
skew_index = high_skew.index

# normalize skewed features using log_transformation
    
for i in skew_index:
    train_test_dummy[i] = np.log1p(train_test_dummy[i])

In [ ]:
# import libraries

from scipy.stats import t, norm
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
# import libraries

from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, KFold
from catboost import CatBoostRegressor
from catboost import Pool 
import shap

In [ ]:
# split data into training and test sets and normalize target 

train_data = train_test_dummy[0:1460]
test_data = train_test_dummy[1460:]
X = train_data
y = np.log1p(target)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

In [ ]:
# SalePrice before transformation

fig, ax = plt.subplots(1,2, figsize= (15,5))
fig.suptitle(" qq-plot & distribution SalePrice ", fontsize= 15)

sm.qqplot(target, stats.t, distargs=(4,),fit=True, line="45", ax = ax[0])

sns.distplot(target, kde = True, hist=True, fit = norm, ax = ax[1])
plt.show()

In [ ]:
# SalePrice before transformation

fig, ax = plt.subplots(1,2, figsize= (15,5))
fig.suptitle(" qq-plot & distribution SalePrice ", fontsize= 15)

sm.qqplot(y, stats.t, distargs=(4,),fit=True, line="45", ax = ax[0])

sns.distplot(y, kde = True, hist=True, fit = norm, ax = ax[1])
plt.show()

In [ ]:
# fit model and evaluate metrics

def root_mean_squared_log_error(y_true, y_pred):
    return np.sqrt(metrics.mean_squared_log_error(y_true, y_pred))

params = {'iterations': 6000,
          'learning_rate': 0.005,
          'depth': 4,
          'l2_leaf_reg': 1,
          'eval_metric':'RMSE',
          'early_stopping_rounds': 200,
          'verbose': 200,
          'random_seed': 42}

cb = CatBoostRegressor(**params)
cb.fit(X_train, y_train,
       eval_set=(X_test, y_test),
       verbose=0)

y_train_predict = np.expm1(cb.predict(X_train))
y_test_predict = np.expm1(cb.predict(X_test))
print(round(root_mean_squared_log_error(np.expm1(y_train), y_train_predict), 3))
print(round(root_mean_squared_log_error(np.expm1(y_test), y_test_predict), 3))

In [ ]:
# feature importance Interactive Plot 

train_pool = Pool(X_train)
test_pool = Pool(X_test)

explainer = shap.TreeExplainer(cb)
shap_values = explainer.shap_values(train_pool)

shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[:200,:], X_train.iloc[:200,:])

In [ ]:
# generate final predictions

cb.fit(X, y)
result = cb.predict(test_data)
result = np.expm1(result)

In [ ]:
# save results to CSV

submission = pd.DataFrame({'Id': id, 'SalePrice': result})
submission.to_csv("submission.csv", index=False)